In [1]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 06, Custom bracket

Companion notebook to [06_custom_bracket.md](06_custom_bracket.md). Define your own rule with `CustomBracket`, declare an axiom profile via flags, and test it through `prove_jacobi`'s generic dispatch path.

## Minimum profile, the commutator rule

`CustomBracket(name, expand_fn, *, degree=..., is_graded_antisymmetric=..., satisfies_leibniz=..., satisfies_graded_jacobi=...)`. The `expand_fn` signature is `(a, b, registry) → Expr`.

In [2]:
from jacopy.brackets.custom import CustomBracket
from jacopy.core.expr import Neg, Product, Sum, Symbol

def commutator(a, b, registry):
    return Sum(Product(a, b), Neg(Product(b, a)))

B = CustomBracket("[·,·]", commutator)
print('name:', B.name, 'degree:', B.degree)
print('antisym:', B.is_graded_antisymmetric)
print('expand X,Y:', B(Symbol('X'), Symbol('Y')).expand())

name: [·,·] degree: 0
antisym: True
expand X,Y: ((X * Y) + (-(Y * X)))


## Axiom flags

Override the defaults with flags when your bracket has a different axiom profile. `satisfies_graded_jacobi=None` is **conditional** Jacobi (as on a derived bracket).

In [3]:
B_asym = CustomBracket(
    "asym",
    lambda a, b, reg: Product(a, b),
    is_graded_antisymmetric=False,
    satisfies_leibniz=False,
    satisfies_graded_jacobi=False,
)
print('antisym:', B_asym.is_graded_antisymmetric,
      'leibniz:', B_asym.satisfies_leibniz,
      'jacobi:', B_asym.satisfies_graded_jacobi)

antisym: False leibniz: False jacobi: False


## `prove_jacobi`, generic dispatch

For the commutator rule the chain is bracket-expand → simplify → 0. A wrong / asymmetric rule leaves a residual and raises `ProofFailure`.

In [4]:
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.verifier import prove_jacobi
from jacopy.proof.strategies import ProofFailure

reg = PropertyRegistry()
for s in (Symbol('X'), Symbol('Y'), Symbol('Z')):
    reg.declare(s, Graded(degree=0))

chain = prove_jacobi(B, Symbol('X'), Symbol('Y'), Symbol('Z'), registry=reg)
print('commutator chain len:', len(chain))
for st in chain.steps:
    print(' ', st.rule)
print('final:', chain.steps[-1].after)

try:
    prove_jacobi(B_asym, Symbol('X'), Symbol('Y'), Symbol('Z'), registry=reg)
except ProofFailure as exc:
    print('\nasym rule fails (as expected):')
    print(' ', str(exc)[:110])

commutator chain len: 2
  bracket-expand
  simplify
final: 0

asym rule fails (as expected):
  ExpandAndSimplify left residual (3 * X * Y * Z) when proving ((X * (Y * Z)) + (Y * (Z * X)) + (Z * (X * Y))) =


## Axiom-obstruction helpers

Inherited from `GradedBracket`: each helper returns the explicit Expr asserted by the axiom. Useful for probing a rule without going into a full proof.

In [5]:
a, b, c = Symbol('a'), Symbol('b'), Symbol('c')
for s in (a, b, c):
    reg.declare(s, Graded(degree=0))

print('antisym obs:', B.graded_antisymmetry_obstruction(a, b, reg))
print('jacobi obs :', B.graded_jacobi_obstruction(a, b, c, reg))
print('leibniz obs:', B.leibniz_obstruction(a, b, c, reg))

antisym obs: ([·,·](a, b) + [·,·](b, a))
jacobi obs : ([·,·](a, [·,·](b, c)) + [·,·](b, [·,·](c, a)) + [·,·](c, [·,·](a, b)))
leibniz obs: ([·,·](a, (b * c)) + (-([·,·](a, b) * c)) + (-(b * [·,·](a, c))))


## Equality, callable identity

Two `CustomBracket`s compare equal only if they share the same `expand_fn` callable.

In [6]:
rule_a = lambda a, b, reg: Sum(Product(a, b), Neg(Product(b, a)))
rule_b = lambda a, b, reg: Sum(Product(a, b), Product(b, a))
print('same rule:', CustomBracket('B', rule_a) == CustomBracket('B', rule_a))
print('diff rule:', CustomBracket('B', rule_a) == CustomBracket('B', rule_b))

same rule: True
diff rule: False


## Next step

Generator-based automatic bracket construction, [07_derived_bracket.md](07_derived_bracket.md).